In [1]:
import random
import pandas as pd 
from faker import Faker 

fake = Faker("fr_FR")
random.seed(42)
Faker.seed(42)

In [2]:
for _ in range(3):
    print(fake.last_name(), "|", fake.street_address(), "|", fake.city(), "|", fake.phone_number())

Riou | chemin Moulin | Joly | +33 (0)2 32 96 00 13
Rey | 408, boulevard de Humbert | Pons | 0140265423
Ledoux | 2, rue Agathe Barre | Aubry-sur-Bertrand | 02 53 61 84 95


In [3]:
N = 500

entities = []
for i in range(N):
    prefix = random.choice(["Société", "Compagnie", "Etablissements", ""])
    entities.append({
        "true_id": i,
        "name": f"{prefix} {fake.last_name()} et {fake.last_name()}".strip(),
        "address": fake.street_address(),
        "city": fake.city(),
        "phone": fake.phone_number()
    })

crm = pd.DataFrame(entities)
crm.insert(0, "id", [f"A{i}" for i in range(len(crm))])
crm.head()

,id,true_id,name,address,city,phone
0,A0,0,Société Verdier et Olivier,"1, rue de Leleu",Chevalier,+33 2 40 64 75 25
1,A1,1,Société Monnier et Lacroix,"31, rue de Goncalves",Gillet-la-Forêt,+33 4 75 03 05 64
2,A2,2,Etablissements François et Lecomte,"83, avenue Guy Aubry",Hoarau,0567965328
3,A3,3,Compagnie Teixeira et Robin,rue Techer,Bazin,01 83 66 97 84
4,A4,4,Compagnie Lenoir et Dubois,"9, chemin Frédéric Cousin",Barbierdan,+33 (0)4 11 82 81 48


In [4]:
ABBR = {"Société": "Ste", "Compagnie": "Cie", "Etablissements": "Ets", "et": "&"}

def noisy(text):
    # 1. Abréviations (une chance sur deux pour chacune)
    for long, court in ABBR.items():
        if random.random() < 0.5:
            text = text.replace(long, court)

    # 2. Faute de frappe : on inverse deux lettres voisines (30 % des cas)
    if random.random() < 0.3 and len(text) > 4:
        i = random.randint(1, len(text) - 2)
        text = text[:i] + text[i + 1] + text[i] + text[i + 2:]

    # 3. Tout en MAJUSCULES (30 % des cas)
    if random.random() < 0.3:
        text = text.upper()

    return text

# Test : on abîme 5 fois le même nom
for _ in range(5):
    print(noisy("Société Dupont et Martin"))

Ste Dupont & Martin
SOCIÉTÉ DUPONT & MRATIN
Scoiété Dupont et Martin
Ste Dupont & Martin
Ste Dupont & Martin


In [ ]:
rows = []

for e in random.sample(entities, int(N * 0.7)):
    rows.append({
        "true_id": e["true_id"],
        "name": noisy(e["name"]),
        "address": noisy(e["address"]),
        "city": e["city"] if random.random() > 0.1 else e["city"].lower(),
        "phone": e["phone"].replace(" ", "") if random.random() < 0.5 else "",
    })

for _ in range(100):
    rows.append({
        "true_id": -1,
        "name": fake.company(),
        "address": fake.street_address(),
        "city": fake.city(),
        "phone": fake.phone_number(),
    })

erp = pd.DataFrame(rows).sample(frac=1, random_state=1).reset_index(drop=True)
erp.insert(0, "id", [f"B{i}" for i in range(len(erp))])

print(len(erp), "lignes dans l'ERP")
erp.head(10)

450 lignes dans l'ERP


,id,true_id,name,address,city,phone
0,B0,-1,Ledoux,"38, rue Éric Morin",Sainte Alain,+33 1 55 98 19 60
1,B1,-1,Guillou Chevalier S.A.R.L.,"34, rue de Moulin",Schmitt-sur-Mer,0438973471
2,B2,51,Et sBoulanger et Parent,"85, rue de Maurice",Saint Louise-la-Forêt,+33234611724
3,B3,14,Ste Gomes & aHmel,AVENUE LOISEAU,Roussel-sur-Legrand,+33413251354
4,B4,6,Société Chevalier & Clément,"91, rue Delams",Dias-les-Bains,0563383465
5,B5,71,Ets Peltiere t Lamy,"576, boulevar dNoël",Perret,
6,B6,413,Ste Renaud & Leroux,"50, boulevadr Joseph Weiss",Sainte Sabine,+33(0)179159416
7,B7,-1,Guilbert SARL,rue Guillou,Marchal,+33 5 63 03 40 84
8,B8,375,Ets Charrier et Laine,BOULEVARD MARGAR& ROSSI,Sauvage,
9,B9,244,Etablissements Blanchard & Denis,boulevard Godard,TanguyBourg,


In [8]:
tid = 14
pd.concat([
    crm[crm.true_id == tid].assign(source="CRM"),
    erp[erp.true_id == tid].assign(source="ERP")
])

,id,true_id,name,address,city,phone,source
14,A14,14,Société Gomes et Hamel,avenue Loiseau,Roussel-sur-Legrand,+33 4 13 25 13 54,CRM
3,B3,14,Ste Gomes & aHmel,AVENUE LOISEAU,Roussel-sur-Legrand,+33413251354,ERP


In [9]:
# 2. Sauvegarde en CSV, comme si on recevait deux exports de logiciels
import os
os.makedirs("data", exist_ok=True)
crm.to_csv("data/source_crm.csv", index=False)
erp.to_csv("data/source_erp.csv", index=False)
print("Fichiers sauvegardés :", os.listdir("data"))

Fichiers sauvegardés : ['source_erp.csv', 'source_crm.csv']


In [10]:
import re
import unicodedata

ABBR_NORM = {
    r"\bste\b": "societe",
    r"\bcie\b": "compagnie",
    r"\bets\b": "etablissements",
    r"&": " et ",
}

def normalize(text):
    text = str(text or "").lower()                                    # 1. minuscules
    text = unicodedata.normalize("NFKD", text)                        # 2. sépare "é" en "e" + accent
    text = text.encode("ascii", "ignore").decode()                    #    puis supprime l'accent
    for pattern, remplacement in ABBR_NORM.items():                   # 3. développe les abréviations
        text = re.sub(pattern, remplacement, text)
    text = re.sub(r"[^a-z0-9 ]", " ", text)                           # 4. ponctuation -> espace
    text = re.sub(r"\s+", " ", text).strip()                          # 5. un seul espace
    return text

tests = ["SOCIÉTÉ DUPONT & MRATIN", "Ste Dupont & Martin", "Société Dupont et Martin",
         "Ets Peltiere t Lamy", "Stephane Petit"]
for t in tests:
    print(f"{t:30} -> {normalize(t)}")

SOCIÉTÉ DUPONT & MRATIN        -> societe dupont et mratin
Ste Dupont & Martin            -> societe dupont et martin
Société Dupont et Martin       -> societe dupont et martin
Ets Peltiere t Lamy            -> etablissements peltiere t lamy
Stephane Petit                 -> stephane petit


In [11]:
from sqlalchemy import create_engine
eng = create_engine("postgresql://er:er@localhost:5432/er")

crm = pd.read_csv("data/source_crm.csv", dtype=str).fillna("")
erp = pd.read_csv("data/source_erp.csv", dtype=str).fillna("")

for df in (crm, erp):
    df["text"] = (df["name"] + " " + df["address"] + " " + df["city"]).map(normalize)

crm.to_sql("raw_crm", eng, if_exists="replace", index=False)
erp.to_sql("raw_erp", eng, if_exists="replace", index=False)   

pd.read_sql("SELECT id, name, text FROM raw_erp LIMIT 5", eng)

,id,name,text
0,B0,Ledoux,ledoux 38 rue eric morin sainte alain
1,B1,Guillou Chevalier S.A.R.L.,guillou chevalier s a r l 34 rue de moulin sch...
2,B2,Et sBoulanger et Parent,et sboulanger et parent 85 rue de maurice sain...
3,B3,Ste Gomes & aHmel,societe gomes et ahmel avenue loiseau roussel ...
4,B4,Société Chevalier & Clément,societe chevalier et clement 91 rue delams dia...


In [12]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
import numpy as np

embed = DefaultEmbeddingFunction()

textes = [
    "societe dupont et martin 12 rue de la paix paris",   # original
    "societe dupont et mratin 12 rue de la paix paris",   # même entreprise avec une faute
    "boulangerie leroy 4 avenue victor hugo lyon"
]

vecteurs = np.array(embed(textes))
print("Forme :", vecteurs.shape)           # (3 textes, 384 nombres)
print("Début du 1er vecteur :", vecteurs[0][:5])

/Users/onizukalto/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 27.5MiB/s]


Forme : (3, 384)
Début du 1er vecteur : [-0.02080906  0.03336485 -0.01465241 -0.03220413 -0.01324012]


In [14]:
def similarite(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("original vs faute    :", round(similarite(vecteurs[0], vecteurs[1]), 3))
print("original vs autre    :", round(similarite(vecteurs[0], vecteurs[2]), 3))

original vs faute    : 0.95
original vs autre    : 0.375


In [15]:
import chromadb

client = chromadb.PersistentClient(path="data/chroma")

try:
    client.delete_collection("crm")
except Exception:
    pass

col = client.create_collection(
    "crm",
    embedding_function=embed,
    metadata={"hnsw:space": "cosine"}
)

col.add(ids=crm["id"].tolist(), documents=crm["text"].tolist())
print(col.count(), "fiches CRM indexées")

500 fiches CRM indexées


In [16]:
fiche = erp.iloc[3]
print("Fiche ERP :", fiche["name"], "| true_id =", fiche["true_id"])

res = col.query(query_texts=[fiche["text"]], n_results=5)

crm_idx = crm.set_index("id")
candidats = crm_idx.loc[res["ids"][0], ["true_id", "name", "city"]]
candidats["distance"] = res["distances"][0]
candidats

Fiche ERP : Ste Gomes & aHmel | true_id = 14


,true_id,name,city,distance
id,,,,
A14,14,Société Gomes et Hamel,Roussel-sur-Legrand,0.022636
A31,31,Société Loiseau et Brunet,Gosselin,0.260969
A115,115,Société Rousset et Ramos,Sainte Simone,0.275085
A181,181,Société Blanchet et Girard,Bouvet-sur-Mer,0.289984
A258,258,Société Adam et Martel,Roche,0.295440


In [27]:
TOP_K = 5

res = col.query(query_texts=erp["text"].tolist(), n_results=TOP_K)

trouve = []
for i, fiche in erp.iterrows():
    if fiche["true_id"] == "-1":    
        continue
    true_ids_candidats = crm_idx.loc[res["ids"][i], "true_id"].values
    trouve.append(fiche["true_id"] in true_ids_candidats)

recall = sum(trouve) / len(trouve)
print(f"Recall du blocking (top {TOP_K}) : {recall:.1%}  ({sum(trouve)}/{len(trouve)})")

Recall du blocking (top 5) : 99.7%  (349/350)


In [28]:
for i, fiche in erp.iterrows():
    if fiche["true_id"] == "-1":
        continue
    if fiche["true_id"] not in crm_idx.loc[res["ids"][i], "true_id"].values:
        print("=== Fiche ERP ratée ===")
        print(fiche[["id", "true_id", "name", "address", "city"]].to_string())
        print("\n=== La vraie fiche CRM ===")
        print(crm[crm.true_id == fiche["true_id"]][["id", "name", "address", "city"]].to_string(index=False))
        print("\n=== Ce que ChromaDB a proposé à la place ===")
        print(crm_idx.loc[res["ids"][i][:3], ["true_id", "name", "address", "city"]].to_string())

=== Fiche ERP ratée ===
id                                  B379
true_id                                5
name       Compagnie Ryanaud et Philippe
address                92, rue de Joseph
city                               Camus

=== La vraie fiche CRM ===
id                          name           address  city
A5 Compagnie Raynaud et Philippe 92, rue de Joseph Camus

=== Ce que ChromaDB a proposé à la place ===
     true_id                            name                   address            city
id                                                                                    
A154     154      Compagnie Bruneau et Jacob      97, rue de Berthelot           Camus
A196     196     Compagnie Pereira et Michel  24, rue Thibault Chauvin  Camus-la-Forêt
A233     233  Compagnie Rodrigues et Vincent  4, rue Sébastien Normand       Rodrigues


In [29]:
fiche = erp[erp.id == "B379"].iloc[0]

v_erp = np.array(embed([fiche["text"]]))[0]
v_crm = np.array(embed(crm["text"].tolist()))

distances = [1 - similarite(v_erp, v) for v in v_crm]
crm_exact = crm.assign(distance=distances).sort_values("distance")

print("Distance avec la vraie fiche A5 :", round(crm_exact.set_index("id").loc["A5", "distance"], 4))
crm_exact[["id", "name", "address", "city", "distance"]].head(5)

Distance avec la vraie fiche A5 : 0.3509


,id,name,address,city,distance
154,A154,Compagnie Bruneau et Jacob,"97, rue de Berthelot",Camus,0.219908
196,A196,Compagnie Pereira et Michel,"24, rue Thibault Chauvin",Camus-la-Forêt,0.245502
233,A233,Compagnie Rodrigues et Vincent,"4, rue Sébastien Normand",Rodrigues,0.284712
320,A320,Compagnie Rodrigues et David,"694, rue Nicolas",Colin-les-Bains,0.285304
221,A221,Compagnie Alves et Labbé,"21, rue Dos Santos",Boulanger,0.306057


In [30]:
from rapidfuzz import fuzz

print("ratio            raynaud / ryanaud          :", fuzz.ratio("raynaud", "ryanaud"))
print("ratio            dupont martin / martin dupont :", fuzz.ratio("dupont martin", "martin dupont"))
print("token_sort_ratio dupont martin / martin dupont :", fuzz.token_sort_ratio("dupont martin", "martin dupont"))
print("ratio            12 rue paix / 12 rue de la paix paris :", fuzz.ratio("12 rue paix", "12 rue de la paix paris"))
print("token_set_ratio  12 rue paix / 12 rue de la paix paris :", fuzz.token_set_ratio("12 rue paix", "12 rue de la paix paris"))

ratio            raynaud / ryanaud          : 85.71428571428572
ratio            dupont martin / martin dupont : 46.15384615384615
token_sort_ratio dupont martin / martin dupont : 100.0
ratio            12 rue paix / 12 rue de la paix paris : 64.70588235294117
token_set_ratio  12 rue paix / 12 rue de la paix paris : 100.0


In [31]:
import re

def digits(s):
    """Garde les 9 derniers chiffres du téléphone : +33 1 23... et 01 23... deviennent identiques."""
    return re.sub(r"\D", "", str(s or ""))[-9:]

def pair_features(rec, cand, dist):
    return {
        "name_score": fuzz.token_sort_ratio(normalize(rec["name"]), normalize(cand["name"])) / 100,
        "addr_score": fuzz.token_set_ratio(normalize(rec["address"]), normalize(cand["address"])) / 100,
        "city_score": fuzz.ratio(normalize(rec["city"]), normalize(cand["city"])) / 100,
        "phone_eq":   float(digits(rec["phone"]) != "" and digits(rec["phone"]) == digits(cand["phone"])),
        "emb_dist":   dist,
    }

rows = []
for i, fiche in erp.iterrows():
    for cid, dist in zip(res["ids"][i], res["distances"][i]):
        cand = crm_idx.loc[cid]
        rows.append({
            "erp_id": fiche["id"],
            "crm_id": cid,
            "label": int(fiche["true_id"] == cand["true_id"]),  
            **pair_features(fiche, cand, dist),
        })

pairs = pd.DataFrame(rows)
print(len(pairs), "paires candidates,", pairs.label.sum(), "vrais matchs")
pairs.head(10)

2250 paires candidates, 349 vrais matchs


,erp_id,crm_id,label,name_score,addr_score,city_score,phone_eq,emb_dist
0,B0,A458,0,0.380952,0.318182,0.235294,0.0,0.261243
1,B0,A446,0,0.216216,0.486486,0.285714,0.0,0.293806
2,B0,A46,0,0.205128,0.380952,0.285714,0.0,0.325091
3,B0,A439,0,0.240000,0.523810,0.560000,0.0,0.330351
4,B0,A499,0,0.363636,0.571429,0.210526,0.0,0.340463
5,B1,A0,0,0.352941,0.600000,0.416667,0.0,0.297357
6,B1,A199,0,0.528302,0.476190,0.296296,0.0,0.299317
7,B1,A87,0,0.545455,0.457143,0.222222,0.0,0.301651
8,B1,A6,0,0.528302,0.551724,0.275862,0.0,0.337274
9,B1,A299,0,0.490566,0.411765,0.666667,0.0,0.339543


In [32]:
pairs.groupby("label")[["name_score", "addr_score", "city_score", "phone_eq", "emb_dist"]].mean().round(2)

,name_score,addr_score,city_score,phone_eq,emb_dist
label,,,,,
0,0.48,0.51,0.39,0.00,0.32
1,0.96,0.98,1.00,0.49,0.04


In [33]:
from sklearn.model_selection import train_test_split

FEATURES = ["name_score", "addr_score", "city_score", "phone_eq", "emb_dist"]
X = pairs[FEATURES]
y = pairs["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0
)
print("Entraînement :", len(X_train), "paires,", y_train.sum(), "vrais matchs")
print("Test         :", len(X_test), "paires,", y_test.sum(), "vrais matchs")

Entraînement : 1575 paires, 244 vrais matchs
Test         : 675 paires, 105 vrais matchs


In [34]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0)
clf.fit(X_train, y_train)        

pred = clf.predict(X_test)        

print(classification_report(y_test, pred, digits=3))
print("Matrice de confusion :")
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0      1.000     1.000     1.000       570
           1      1.000     1.000     1.000       105

    accuracy                          1.000       675
   macro avg      1.000     1.000     1.000       675
weighted avg      1.000     1.000     1.000       675

Matrice de confusion :
[[570   0]
 [  0 105]]


In [37]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import precision_score, recall_score, f1_score

mlflow.set_tracking_uri("http://localhost:5001")   
mlflow.set_experiment("entity-resolution")   

params = {"n_estimators": 200, "max_depth": 6, "top_k": TOP_K}

with mlflow.start_run(run_name="random_forest"):
    clf = RandomForestClassifier(n_estimators=params["n_estimators"],
                                 max_depth=params["max_depth"], random_state=0)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    metrics = {
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "blocking_recall": recall,          # calculé à la cellule 12
    }
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(clf, name="model", serialization_format="cloudpickle")
    
print(metrics)

2026/10/18 21:05:49 INFO mlflow.tracking.fluent: Experiment with name 'entity-resolution' does not exist. Creating a new experiment.
2026/10/18 21:05:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run random_forest at: http://localhost:5001/#/experiments/1/runs/a52920b18d32402985ea9ecca296a360
🧪 View experiment at: http://localhost:5001/#/experiments/1
{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'blocking_recall': 0.9971428571428571}


In [38]:
for depth in [1, 2, 6]:
    with mlflow.start_run(run_name=f"rf_depth_{depth}"):
        clf_test = RandomForestClassifier(n_estimators=200, max_depth=depth, random_state=0)
        clf_test.fit(X_train, y_train)
        p = clf_test.predict(X_test)
        mlflow.log_params({"n_estimators": 200, "max_depth": depth, "top_k": TOP_K})
        mlflow.log_metrics({
            "precision": precision_score(y_test, p),
            "recall": recall_score(y_test, p),
            "f1": f1_score(y_test, p),
        })
        print(f"max_depth={depth} -> F1 = {f1_score(y_test, p):.3f}")

max_depth=1 -> F1 = 1.000
🏃 View run rf_depth_1 at: http://localhost:5001/#/experiments/1/runs/c076f8d557b54ca6bce8e34b37a043ca
🧪 View experiment at: http://localhost:5001/#/experiments/1
max_depth=2 -> F1 = 1.000
🏃 View run rf_depth_2 at: http://localhost:5001/#/experiments/1/runs/8db2420c377c485c8fec4a88fca2cfec
🧪 View experiment at: http://localhost:5001/#/experiments/1
max_depth=6 -> F1 = 1.000
🏃 View run rf_depth_6 at: http://localhost:5001/#/experiments/1/runs/895a52c1ca1a41698486d212a802f5a7
🧪 View experiment at: http://localhost:5001/#/experiments/1


In [39]:
pairs["proba"] = clf.predict_proba(pairs[FEATURES])[:, 1]

best = pairs.sort_values("proba", ascending=False).drop_duplicates("erp_id")

SEUIL = 0.5
matches = best[best["proba"] >= SEUIL][["erp_id", "crm_id", "proba"]]

print(f"{len(matches)} fiches ERP rattachées au CRM, sur {len(erp)}")
print(f"Attendu : {(erp.true_id != '-1').sum()} (les autres sont des entreprises inconnues)")

matches.to_sql("matches", eng, if_exists="replace", index=False)

349 fiches ERP rattachées au CRM, sur 450
Attendu : 350 (les autres sont des entreprises inconnues)


349

In [40]:
verif = pd.read_sql("""
    SELECT
        m.erp_id,
        e.name      AS nom_erp,
        c.name      AS nom_crm,
        m.proba,
        (e.true_id = c.true_id) AS correct
    FROM matches m
    JOIN raw_erp e ON e.id = m.erp_id
    JOIN raw_crm c ON c.id = m.crm_id
    ORDER BY m.proba ASC
""", eng)

print("Rattachements corrects :", verif["correct"].sum(), "/", len(verif))
print("Fiche ERP manquante    :", set(erp[erp.true_id != "-1"].id) - set(verif.erp_id))

verif.head(10)   # les 10 matchs dont le modèle est le MOINS sûr

Rattachements corrects : 349 / 349
Fiche ERP manquante    : {'B379'}


,erp_id,nom_erp,nom_crm,proba,correct
0,B2,Et sBoulanger et Parent,Etablissements Boulanger et Parent,0.845,True
1,B350,MAILLARD ET REY,Maillard et Rey,0.850,True
2,B415,Ste Bouch& & iBgot,Société Bouchet et Bigot,0.875,True
3,B38,Ste Pottier et Reynadu,Société Pottier et Reynaud,0.880,True
4,B109,Sociét éMathieu & Mace,Société Mathieu et Mace,0.910,True
5,B217,Compagnie Meunier & Chauv&,Compagnie Meunier et Chauvet,0.930,True
6,B362,Jean & eLmoine,Jean et Lemoine,0.930,True
7,B212,Compagnie Dias et Gillte,Compagnie Dias et Gillet,0.945,True
8,B27,Nicolas et Boulay,Nicolas et Boulay,0.955,True
9,B348,Gomes & Thoams,Gomes et Thomas,0.960,True


In [41]:
import requests

r = requests.post("http://localhost:8000/match", json={
    "name": "STE RAY & BDOIN",
    "address": "9 bd umber",
    "city": "pson",
})
resultat = r.json()
print("Doublon :", resultat["doublon"])
print("Meilleur candidat :", resultat["meilleur_candidat"])

Doublon : False
Meilleur candidat : {'crm_id': 'A257', 'name': 'Société Rodriguez et Baudry', 'city': 'Saint Raymond', 'proba': 0.0}


In [42]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

eng = create_engine("postgresql+psycopg2://er:er@localhost:5432/er")
FEATURES = ["name_score", "addr_score", "city_score", "phone_eq", "emb_dist"]

pairs = pd.read_sql("SELECT * FROM candidate_pairs", eng)
crm = pd.read_sql("SELECT id, name, address, city, phone FROM raw_crm", eng)
erp = pd.read_sql("SELECT id, name, address, city, phone FROM raw_erp", eng)

# Même découpe et même modèle que dans le pipeline (random_state identiques)
X_tr, X_te, y_tr, y_te = train_test_split(pairs[FEATURES], pairs["label"],
                                          test_size=0.3, stratify=pairs["label"], random_state=0)
clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0).fit(X_tr, y_tr)

test = pairs.loc[X_te.index].copy()
test["pred"] = clf.predict(X_te)
test["proba"] = clf.predict_proba(X_te)[:, 1].round(2)

# Les entreprises du CRM qui ont un homonyme (même nom, même ville)
homonymes = set(crm[crm.duplicated(["name", "city"], keep=False)].id)
test["crm_a_un_homonyme"] = test.crm_id.isin(homonymes)

def detail(df):
    return (df.merge(erp.add_suffix("_erp"), left_on="erp_id", right_on="id_erp")
              .merge(crm.add_suffix("_crm"), left_on="crm_id", right_on="id_crm")
              [["name_erp", "name_crm", "address_erp", "address_crm", "city_erp", "city_crm",
                "phone_erp", "proba", "crm_a_un_homonyme"] + FEATURES])

faux_positifs = detail(test[(test.pred == 1) & (test.label == 0)])
faux_negatifs = detail(test[(test.pred == 0) & (test.label == 1)])
print(len(faux_positifs), "faux positifs |", len(faux_negatifs), "faux négatifs")

4 faux positifs | 9 faux négatifs


In [43]:
faux_positifs

,name_erp,name_crm,address_erp,address_crm,city_erp,city_crm,phone_erp,proba,crm_a_un_homonyme,name_score,addr_score,city_score,phone_eq,emb_dist
0,Société Marie& Sanchez,Société Marie et Sanchez,"18, BOULEVARD FRANÇOIS ECRON","6, boulevard Huet",RodriguezVille,RodriguezVille,0495053315,0.76,True,1.000000,0.720000,1.0,0.0,0.087866
1,St Delorme ez Michel,Société Delorme et Michel,"54, boulevard de Delpas","466, boulevard Blot",Henry,Henry,0466611952,0.63,True,0.844444,0.666667,1.0,0.0,0.286096
2,Société Devaux et Moreno,Société Jean et Lemoine,"7, avenue Leconte","809, rue de Mendès",Royernec,Royernec,+33 (0)6 35 87 01 04,0.56,False,0.638298,0.363636,1.0,0.0,0.101044
3,EtablissementslLenoir et Marin,Etablissements Lenoir et Marin,"48, chemin Couturier","7, chemin de Hebert",Roussetdan,Roussetdan,,0.51,True,0.966667,0.540541,1.0,0.0,0.137529


In [44]:
faux_negatifs

,name_erp,name_crm,address_erp,address_crm,city_erp,city_crm,phone_erp,proba,crm_a_un_homonyme,name_score,addr_score,city_score,phone_eq,emb_dist
0,Toussaint et Balily,Toussaint et Bailly,boulevard de Delannoy,"83, rue Roger Teixeira",Lacroix-sur-Robert,Lacroix-sur-Robert,,0.13,False,0.947368,0.380952,1.000000,0.0,0.255091
1,Ets Chauveau & qlancahrd,Etablissements Chauveau et Blanchard,"7, bd de Bzrrtand","7, boulevard de Bertrand",Picard,Picard,,0.21,False,0.722222,0.564103,1.000000,0.0,0.232260
2,et Delorme,Traore et Delorme,"661, rue de Étienne","1, rue de Camus",Sainte aLure,Sainte Laure,,0.32,False,0.740741,0.600000,0.916667,0.0,0.249747
3,Etablissements Lisefu & Brunet,Etablissements Loiseau et Brunet,"58, avenue Descamps","27, rue de Turpin",Gosselin,Gosselin,,0.19,False,0.952381,0.352941,1.000000,0.0,0.249201
4,Mathieu & Cols,Mathieu et Colas,"24, avenue Pénélope Bouvier","290, boulevard Geneviève Barbe",Bertin,Bertin,,0.32,False,0.967742,0.545455,1.000000,0.0,0.255771
5,STEC LERC & AUBRY,Société Clerc et Aubry,"789, boulevard Gilbert Poirier",rue Barthelemy,Ollivier-la-Forêt,Ollivier-la-Forêt,,0.04,False,0.700000,0.372093,1.000000,0.0,0.297221
6,Compagnie Legros d Lcoq,Compagnie Legros et Lecoq,"1, rueAlexandre Mgrques","1, rue Alexandre Marques",Bodin,Bodin,,0.38,True,0.916667,0.533333,1.000000,0.0,0.168258
7,EtablissementslLenoir et Marin,Etablissements Lenoir et Marin,"48, chemin Couturier",boulevard Thibaut Teixeira,Roussetdan,Roussetdan,,0.31,True,0.966667,0.266667,1.000000,0.0,0.319087
8,ETABLISSEMENTS GOMES & THOMAS,Etablissements Gomes et Thomas,"38, rue Paul Ribeiro","25, rue de Gallet",Blot,Blot,,0.47,False,1.000000,0.457143,1.000000,0.0,0.116819
